Paquetes necesarios

In [1]:
import random
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

TAREA: Realiza la cuenta de píxeles blancos por filas (en lugar de por columnas). Determina el valor máximo de píxeles blancos para filas, maxfil, mostrando el número de filas y sus respectivas posiciones, con un número de píxeles blancos mayor o igual que 0.90*maxfil. Resalta con alguna primitiva gráfica en la imagen de Canny las filas que cumplen dicha condición.

In [ ]:
# Leemos la imagen del archivo
img = cv2.imread('images/mandril.jpg')

# Si hay lectura correcta
if img is not None:
    # Conversión de la imagen a niveles de grises de la imagen original en BGR
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Obtenemos los contornos con el operador de Canny
    canny = cv2.Canny(gris, 100, 200)

    # Contamos el número de píxeles blancos (255) por fila
    # Sumamos los valores de los pixeles por fila
    row_counts = cv2.reduce(canny, 1, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)
    row_counts = row_counts.flatten()  # Aplanamos el array para que sea unidimensional

    # Normalizamos en base al número de columnas, segundo valor devuelto por shape, y al valor máximo del píxel (255)
    # El resultado será el número de píxeles blancos por fila
    rows = row_counts / (255 * canny.shape[1])

    # Obtenemos el valor máximo de píxeles blancos para filas
    maxfil = rows.max()

    # Obtenemos el índice de la fila con el máximo número de píxeles blancos
    idx_maxfil = np.where(rows == maxfil)[0][0]  

    # Creamos una lista para almacenar los índices de las filas que cumplen el umbral
    idx_rows_threshold = []

    # Mostramos dicha cuenta gráficamente
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.axis("off")
    plt.title("Canny")
    for i in range(len(rows)):
        if rows[i] >= maxfil*0.9:
            idx_rows_threshold.append(i)                                 # Añadimos el índice de la fila que cumple el umbral a la lista

            if i == idx_maxfil: 
                plt.axhline(y=i, color='g', linestyle='-', linewidth=1)  # Mostramos la fila con el máximo número de píxeles blancos
            else:
                plt.axhline(y=i, color='r', linestyle='-', linewidth=1)  # Mostramos las filas que superan el 90% del valor máximo
    plt.imshow(canny, cmap='gray')

    # Calculamos el umbral como el 90% del valor máximo de píxeles blancos por fila
    umbral = maxfil * 0.9  

    plt.subplot(1, 2, 2)
    plt.title("Respuesta de Canny")
    plt.xlabel("Filas")
    plt.ylabel("% píxeles")
    plt.plot(rows, label='Porcentaje de píxeles blancos por fila')
    plt.scatter(idx_maxfil, maxfil, color='g', zorder=5, label=f'maxfil = {maxfil:.2f}')  # Marcamos el máximo con un punto verde
    plt.axhline(y=umbral, color='r', linestyle='--', label=f'0.90*maxfil = {umbral:.2f}')  # Línea horizontal para el umbral
    # Rango en x definido por las filas
    plt.xlim([0, canny.shape[0]])
    plt.legend()
    plt.tight_layout(pad=2.0, w_pad=3.0) # Ajustamos el espaciado entre subplots

    plt.savefig('./images/exercise1.png',
            bbox_inches='tight',
            pad_inches=0.3)

    print(f"Hay {len(idx_rows_threshold)} filas con un número de píxeles blancos mayor o igual que 0.90*maxfil y estas son sus posiciones: {idx_rows_threshold}")

else: 
    print('Imagen no encontrada')

TAREA: Aplica umbralizado a la imagen resultante de Sobel (convertida a 8 bits), y posteriormente realiza el conteo por filas y columnas similar al realizado en el ejemplo con la salida de Canny de píxeles no nulos. Calcula el valor máximo de la cuenta por filas y columnas, y determina las filas y columnas por encima del 0.90*máximo. Remarca con alguna primitiva gráfica dichas filas y columnas sobre la imagen del mandril. Visualiza los resultados obtenidos para la imagen (o una de tu elección) con Canny y Sobe tras umbralizar ¿Cómo se comparan los resultados obtenidos a partir de Sobel y Canny?

In [ ]:
# Funciones auxiliares para el cálculo de filas y columnas, resaltado y graficado

def get_rows_columns(image):
    # Contamos el número de píxeles blancos (255) por fila
    # Sumamos los valores de los pixeles por fila
    row_counts = cv2.reduce(image, 1, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)

    # Contamos el número de píxeles blancos (255) por columna
    # Sumamos los valores de los pixeles por columna
    col_counts = cv2.reduce(image, 0, cv2.REDUCE_SUM, dtype=cv2.CV_32SC1)

    # Normalizamos en base al número de columnas, segundo valor devuelto por shape, y al valor máximo del píxel (255)
    # El resultado será el número de píxeles blancos por fila
    rows = row_counts[:, 0] / (255 * image.shape[1])

    # Normalizamos en base al número de filas, primer valor devuelto por shape, y al valor máximo del píxel (255)
    # El resultado será el número de píxeles blancos por columna
    cols = col_counts[0, :] / (255 * image.shape[0])

    return rows, cols

def highlight_rows_columns(image, rows, cols, row_color=(0, 0, 255), col_color=(255, 0, 0), max_color=(0, 255, 0)):
    # Convertimos la imagen a BGR para poder dibujar en color
    image_bgr = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    # Obtenemos el valor máximo de píxeles blancos para filas y columnas
    maxfil = np.max(rows)
    maxcol = np.max(cols)

    # Obtenemos el índice de la fila y columna con el valor máximo de píxeles blancos
    idx_maxfil = np.argmax(rows)
    idx_maxcol = np.argmax(cols)

    # Buscamos y resaltamos las filas con un número de píxeles blancos mayor o igual que 0.90*maxfil
    idx_rows = []
    for i in range(0, len(rows)):
        if rows[i] >= (0.90 * maxfil):
            idx_rows.append(i)

            # Si la fila es la que tiene el máximo número de píxeles blancos, la resaltamos en verde, si no, en rojo
            if i == idx_maxfil:
                cv2.line(image_bgr, (0, i), (image_bgr.shape[1], i), max_color, 1)
            else:
                cv2.line(image_bgr, (0, i), (image_bgr.shape[1], i), row_color, 1)

    # Buscamos y resaltamos las columnas con un número de píxeles blancos mayor o igual que 0.90*maxcol
    idx_cols = []
    for i in range(0, len(cols)):
        if cols[i] >= (0.90 * maxcol):
            idx_cols.append(i)

            # Si la columna es la que tiene el máximo número de píxeles blancos, la resaltamos en verde, si no, en azul
            if i == idx_maxcol:
                cv2.line(image_bgr, (i, 0), (i, image_bgr.shape[0]), max_color, 1)
            else:
                cv2.line(image_bgr, (i, 0), (i, image_bgr.shape[0]), col_color, 1)

    return image_bgr, idx_rows, idx_cols

def plot_results(image_bgr, rows, cols, title):
    # Calculamos los valores máximos, sus índices y los umbrales de filas y columnas para construir las gráficas
    maxfil = np.max(rows)
    idx_maxfil = np.argmax(rows)
    umbral_fil = 0.90 * maxfil

    maxcol = np.max(cols)
    idx_maxcol = np.argmax(cols)
    umbral_col = 0.90 * maxcol
    
    # Mostramos el resultado y las cuentas gráficamente
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.axis("off")
    plt.title(f"Imagen: {title}")
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), cmap='gray') 

    plt.subplot(1, 3, 2)
    plt.title(f"Filas - {title}")
    plt.xlabel("Filas")
    plt.ylabel("% píxeles")
    plt.plot(rows)
    plt.scatter(idx_maxfil, maxfil, color='g', zorder=5)  # Marcamos el máximo con un punto verde
    plt.axhline(y=umbral_fil, color='r', linestyle='--')  # Línea horizontal para el umbral
    # Rango en x definido por el número de filas de la imagen
    plt.xlim([0, image_bgr.shape[0]])

    plt.subplot(1, 3, 3)
    plt.title(f"Columnas - {title}")
    plt.xlabel("Columnas")
    plt.ylabel("% píxeles")
    plt.plot(cols)
    plt.scatter(idx_maxcol, maxcol, color='g', zorder=5)  # Marcamos el máximo con un punto verde
    plt.axhline(y=umbral_col, color='r', linestyle='--')  # Línea horizontal para el umbral
    # Rango en x definido por el número de columnas de la imagen
    plt.xlim([0, image_bgr.shape[1]])

    plt.tight_layout(pad=3.0, w_pad=4.0) # Separación entre plots

    plt.savefig(f"./images/exercise2_{title}.png",
                    bbox_inches='tight',
                    pad_inches=0.3)
    
    plt.show()

In [ ]:
# Leemos la imagen del archivo
img = cv2.imread('images/mandril.jpg') 

# Si hay lectura correcta
if img is not None:
    # Conversión de la imagen a niveles de grises de la imagen original en BGR
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # --- PROCESO DE CANNY ---

    # Obtenemos los contornos con el operador de Canny
    # Parámetros: imagen de entrada, umbral inferior, umbral superior
    canny = cv2.Canny(gris, 100, 200)

    # Calculamos los píxeles blancos por fila/columna y resaltamos las filas/columnas que cumplen el umbral
    rows_canny, cols_canny = get_rows_columns(canny)
    canny_bgr, idx_rows_canny, idx_cols_canny = highlight_rows_columns(canny, rows_canny, cols_canny)

    print(f"[Canny] Hay {len(idx_rows_canny)} filas con un número de píxeles blancos mayor o igual que 0.90*maxfil y estas son sus posiciones: {idx_rows_canny}")
    print(f"[Canny] Hay {len(idx_cols_canny)} columnas con un número de píxeles blancos mayor o igual que 0.90*maxcol y estas son sus posiciones: {idx_cols_canny}")

    # Mostramos las gráficas para Canny
    plot_results(canny_bgr, rows_canny, cols_canny, "Canny")

    # --- PROCESO DE SOBEL ---

    # Gaussiana para suavizar la imagen original, eliminando altas frecuencias
    ggris = cv2.GaussianBlur(gris, (3, 3), 0)

    # Calcula en ambas direcciones (horizontal y vertical)
    sobelx = cv2.Sobel(ggris, cv2.CV_64F, 1, 0)  # x
    sobely = cv2.Sobel(ggris, cv2.CV_64F, 0, 1)  # y

    # Combina ambos resultados
    sobel = cv2.add(sobelx, sobely)

    # Conversión a byte con openCV
    sobel8 = cv2.convertScaleAbs(sobel)

    # Definimos el valor del umbral
    valorUmbral = 127
    
    # Obtenemos la imagen sobel8 umbralizada para dicho valor definido
    _, imagenSobelUmbralizada = cv2.threshold(sobel8, valorUmbral, 255, cv2.THRESH_BINARY)

    # Calculamos los píxeles blancos por fila/columna y resaltamos las filas/columnas que cumplen el umbral
    rows_sobel, cols_sobel = get_rows_columns(imagenSobelUmbralizada)
    sobel_bgr, idx_rows_sobel, idx_cols_sobel = highlight_rows_columns(imagenSobelUmbralizada, rows_sobel, cols_sobel)

    print(f"[Sobel Umbralizado] Hay {len(idx_rows_sobel)} filas con un número de píxeles blancos mayor o igual que 0.90*maxfil y estas son sus posiciones: {idx_rows_sobel}")
    print(f"[Sobel Umbralizado] Hay {len(idx_cols_sobel)} columnas con un número de píxeles blancos mayor o igual que 0.90*maxcol y estas son sus posiciones: {idx_cols_sobel}")

    # Mostramos las gráficas para Sobel Umbralizado
    plot_results(sobel_bgr, rows_sobel, cols_sobel, "Sobel Umbralizado")

    # --- COMPARATIVA FINAL ---

    # Mostramos la comparación de la imagen de Canny y la imagen de Sobel Umbralizada
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.axis("off")
    plt.title("Resultado Canny")
    plt.imshow(cv2.cvtColor(canny_bgr, cv2.COLOR_BGR2RGB)) 

    plt.subplot(1, 2, 2)
    plt.axis("off")
    plt.title("Resultado Sobel Umbralizado")
    plt.imshow(cv2.cvtColor(sobel_bgr, cv2.COLOR_BGR2RGB))
    plt.tight_layout()

    plt.savefig('./images/exercise2_comparative.png',
                    bbox_inches='tight',
                    pad_inches=0.3)
     
    plt.show()

else:
    print('Imagen no encontrada')


TAREA: Tras ver los vídeos [My little piece of privacy](https://www.niklasroy.com/project/88/my-little-piece-of-privacy), [Messa di voce](https://youtu.be/GfoqiyB1ndE?feature=shared) y [Virtual air guitar](https://youtu.be/FIAmyoEpV5c?feature=shared) proponer un demostrador reinterpretando la parte de procesamiento de la imagen, tomando como punto de partida alguna de dichas instalaciones.

In [2]:
# Función auxiliar para dibujar texto centrado en un frame de OpenCV

def draw_centered_text(frame, text, font, font_scale, color, thickness, top_margin):

    # Dimensiones del frame
    h, w = frame.shape[:2]

    # Obtenemos el ancho y alto del texto en píxeles
    (wide_text, height_text), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    # Calculamos las coordenadas centradas
    x = (w - wide_text) // 2
    y = top_margin + height_text

    # Dibujamos el texto en pantalla
    cv2.putText(frame, text, (x, y), font, font_scale, color, thickness)

In [14]:
# Inicializamos la captura de video desde la cámara web
vid = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# Inicializamos el substractor de fondo MOG2 sin detección de sombras
background_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)

# Variables de tiempo para la calibración del fondo
start_time = time.time()
calibration_seconds = 3

# Propiedades del cuadro de la región de interés (ROI) en píxeles
square_size = 150
x, y = 100, 100     # Coordenadas de la esquina superior izquierda del cuadro
dx, dy = 10, 10     # Velocidad de movimiento del cuadro en píxeles por fotograma

# Gestión del estado inicla del juego
game_state = "MOVING" # Estados posibles: "MOVING", "CAPTURING", "CAUGHT", "SAFE"
state_change_time = time.time() + calibration_seconds + random.randint(3, 6)  # Tiempo de cambio de estado 

while(True):      
    # Leemos fotograma a fotograma
    ret, frame = vid.read()
    if ret:
        # Aplicamos el efecto espejo sobre la entrada
        frame=cv2.flip(frame, 1)

        # Definimos las dimensiones del fotograma
        height, width, _ = frame.shape

        # Calibramos el fondo durante los primeros segundos
        if time.time() - start_time < calibration_seconds:
            # Actualizamos el fondo de la escena sin la presencia del jugador
            foreground_mask = background_subtractor.apply(frame)

            # Indicamos al jugador que permanezca fuera de la escena durante la calibración
            draw_centered_text(frame, "CALIBRANDO: Permanezca fuera de la escena", cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 3, 50)

            # Mostramos la ventana de calibración
            cv2.imshow('Avoid the Box', frame)     # Ventana principal del juego

            # Detenemos pulsando ESC
            if cv2.waitKey(20) == 27:
                break

            continue

        # Congelamos el aprendizaje del fondo (learningRate=0) para que no memorice al jugador si se queda quieto
        foreground_mask = background_subtractor.apply(frame, learningRate=0)

        # Aplicamos umbralización para eliminar el ruido de la máscara de primer plano
        _, foreground_mask = cv2.threshold(foreground_mask, 200, 255, cv2.THRESH_BINARY)

        # Convertimos el fotograma a escala de grises para la detección de bordes
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Obtenemos los bordes del jugador usando el algoritmo de Canny
        edges = cv2.Canny(gray_frame, 100, 200)

        # Filtramos los bordes del jugador usando la máscara para quedarnos solo con la silueta del jugador
        player_edges = cv2.bitwise_and(edges, edges, mask=foreground_mask)

        # Definimos la lógica del juego según el estado actual
        if game_state == "MOVING":
            # Movemos el cuadrado
            x += dx
            y += dy

            # Detectamos las colisiones con los bordes del fotograma y cambiamos la dirección si es necesario
            if x <= 0 or x + square_size >= width: dx = -dx
            if y <= 0 or y + square_size >= height: dy = -dy

            # Comprobamos si es hora de capturar
            if time.time() >= state_change_time:
                game_state = "CAPTURING"
                state_change_time = time.time() + 2  # Capturamos durante 2 segundos

        elif game_state == "CAPTURING":
            # Extraemos la región de interés (ROI) de los bordes del jugador
            square_roi = player_edges[y:y+square_size, x:x+square_size]

            # Contamos el número de píxeles blancos (bordes del jugador) en la ROI
            edge_count = cv2.countNonZero(square_roi)

            # Comprobamos si el jugador ha sido atrapado (más de 50 píxeles blancos denotan que el jugador está dentro del cuadro)
            if edge_count > 50:
                game_state = "CAUGHT"
                state_change_time = time.time() + 3  # Mostramos el mensaje de "CAUGHT" durante 3 segundos
            else:
                # Si se acaba el tiempo de captura y no se ha atrapado al jugador, se salva
                game_state = "SAFE"
                state_change_time = time.time() + 3  # Mostramos el mensaje de "SAFE" durante 3 segundos

        elif game_state in ["CAUGHT", "SAFE"]:
            # Después de mostrar el mensaje, volvemos al estado de movimiento
            if time.time() > state_change_time:
                game_state = "MOVING"
                state_change_time = time.time() + random.randint(3, 6)  # Tiempo aleatorio hasta la próxima captura

        # Traducimos el estado del juego al español para mostrarlo en pantalla
        display_state = ""
        color = (255, 255, 255)  # Color por defecto (blanco)

        # Asignamos el color del cuadrado según el estado actual del juego
        if game_state == "MOVING":
            display_state = "EN MOVIMIENTO"
            color = (255, 255, 0)          # Cyan
        elif game_state == "CAPTURING": 
            display_state = "CAPTURANDO"
            color = (0, 165, 255)          # Naranja
        elif game_state == "CAUGHT": 
            display_state = "ATRAPADO"
            color = (0, 0, 255)            # Rojo 
        elif game_state == "SAFE": 
            display_state = "A SALVO"
            color = (0, 255, 0)            # Verde

        # Dibujamos el cuadrado y el estado del juego en el fotograma
        cv2.rectangle(frame, (x, y), (x + square_size, y + square_size), color, 3)
        cv2.putText(frame, f"Estado: {display_state}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

        # Mostramos las ventanas
        cv2.imshow('Avoid the Box', frame)         # Ventana principal del juego
        cv2.imshow('Player Edges', player_edges)   # Ventana con los bordes del jugador detectados por Canny

        # Detenemos pulsando ESC
        if cv2.waitKey(20) == 27:
            break

# Liberamos el objeto de captura
vid.release()
# Destruimos las ventanas
cv2.destroyAllWindows()

**TAREA ADICIONAL**:

Detección de colores mediante HSV:

In [20]:
import cv2
import numpy as np

class HSVColorDetector:
    def __init__(self, window_name="Configuracion HSV"):
        self.window_name = window_name

        self.h_min = 0
        self.h_max = 179

        self.s_min = 50
        self.s_max = 255

        self.v_min = 50
        self.v_max = 255

        self.width = 700
        self.height = 750

        cv2.namedWindow(self.window_name, cv2.WINDOW_NORMAL)

        cv2.resizeWindow(self.window_name, self.width, self.height)

        cv2.setMouseCallback(self.window_name, self.mouse_callback)

        self.slider_x = 100
        self.slider_width = 500

        self.slider_y = [
            565,  # H min
            600,  # H max
            635,  # S min
            670,  # S max
            705,  # V min
            740   # V max
        ]

        self.active_slider = None

        self.update_preview()

    def mouse_callback(self, event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            for i, slider_y in enumerate(self.slider_y):
                if abs(y - slider_y) < 15:
                    self.active_slider = i
                    self.update_slider(x)
                    return

        elif event == cv2.EVENT_MOUSEMOVE:
            if (self.active_slider is not None and (flags & cv2.EVENT_FLAG_LBUTTON)):
                self.update_slider(x)

        elif event == cv2.EVENT_LBUTTONUP:
            self.active_slider = None

    def update_slider(self, x):
        x = max(self.slider_x, min(self.slider_x + self.slider_width, x))

        position = (x - self.slider_x) / self.slider_width

        values = [179, 179, 255, 255, 255, 255]

        value = round(position * values[self.active_slider])

        if self.active_slider == 0:
            self.h_min = value
            if self.h_min > self.h_max:
                self.h_max = self.h_min

        elif self.active_slider == 1:
            self.h_max = value
            if self.h_max < self.h_min:
                self.h_min = self.h_max

        elif self.active_slider == 2:
            self.s_min = value
            if self.s_min > self.s_max:
                self.s_max = self.s_min

        elif self.active_slider == 3:
            self.s_max = value
            if self.s_max < self.s_min:
                self.s_min = self.s_max

        elif self.active_slider == 4:
            self.v_min = value
            if self.v_min > self.v_max:
                self.v_max = self.v_min

        elif self.active_slider == 5:
            self.v_max = value
            if self.v_max < self.v_min:
                self.v_min = self.v_max

    def create_hue_spectrum(self):
        width = 600
        height = 45

        hue = np.arange(180, dtype=np.uint8)

        hsv = np.zeros((1, 180, 3), dtype=np.uint8)

        hsv[0, :, 0] = hue
        hsv[0, :, 1] = 255
        hsv[0, :, 2] = 255

        spectrum = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        spectrum = cv2.resize(spectrum, (width, height), interpolation=cv2.INTER_NEAREST)

        x_min = int(self.h_min / 179 * (width - 1))
        x_max = int(self.h_max / 179 * (width - 1))

        cv2.line(spectrum, (x_min, 0), (x_min, height - 1), (255, 255, 255), 3)
        cv2.line(spectrum, (x_max, 0), (x_max, height - 1), (255, 255, 255), 3)
        cv2.rectangle(spectrum, (x_min, 0), (x_max, height - 1), (255, 255, 255), 2)

        return spectrum

    def create_sv_square(self):
        size = 300

        h = (self.h_min + self.h_max) // 2
        s = np.linspace(0, 255, size, dtype=np.uint8)
        v = np.linspace(255, 0, size, dtype=np.uint8)

        S, V = np.meshgrid(s, v)

        H = np.full((size, size), h, dtype=np.uint8)

        hsv = np.stack([H, S, V], axis=2)

        sv = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        x1 = int(self.s_min / 255 * (size - 1))
        x2 = int(self.s_max / 255 * (size - 1))

        y1 = int((255 - self.v_max) / 255 * (size - 1))
        y2 = int((255 - self.v_min) / 255 * (size - 1))

        cv2.rectangle( sv, (x1, y1), (x2, y2), (255, 255, 255), 3)

        return sv

    def draw_slider(self, canvas, y, label, value, maximum):
        cv2.putText(canvas, f"{label}: {value}", (100, y - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        x1 = self.slider_x
        x2 = self.slider_x + self.slider_width

        cv2.line(canvas, (x1, y), (x2, y), (130, 130, 130), 4)

        x = int(x1 + (value / maximum) * self.slider_width)

        cv2.circle(canvas, (x, y), 9, (255, 255, 255), -1)
        cv2.circle(canvas, (x, y), 9, (100, 100, 100), 2)

    def update_preview(self):
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)

        cv2.putText(canvas, "Hue (H)", (50, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1)

        spectrum = self.create_hue_spectrum()

        canvas[50:95, 50:650] = spectrum

        cv2.putText(canvas, "0", (50, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(canvas, "179", (620, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        sv = self.create_sv_square()

        canvas[145:445, 50:350] = sv

        cv2.putText(canvas, "Saturation (S)", (50, 475), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        cv2.putText(canvas, "Value (V)", (365, 165), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

        h_mid = (self.h_min + self.h_max) // 2
        s_mid = (self.s_min + self.s_max) // 2
        v_mid = (self.v_min + self.v_max) // 2

        hsv_color = np.uint8([[[h_mid, s_mid, v_mid]]])
        bgr_color = cv2.cvtColor(hsv_color, cv2.COLOR_HSV2BGR)[0, 0]
        color_preview = np.full((130, 220, 3), bgr_color, dtype=np.uint8)
        canvas[365:495, 420:640] = color_preview

        cv2.rectangle(canvas, (420, 365), (640, 495), (255, 255, 255), 2)

        cv2.putText(canvas, "Color seleccionado", (420, 520), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        self.draw_slider(canvas, self.slider_y[0], "H min", self.h_min, 179)
        self.draw_slider(canvas, self.slider_y[1], "H max", self.h_max, 179)
        self.draw_slider(canvas, self.slider_y[2], "S min", self.s_min, 255)
        self.draw_slider(canvas, self.slider_y[3], "S max", self.s_max, 255)
        self.draw_slider(canvas, self.slider_y[4], "V min", self.v_min,255)
        self.draw_slider(canvas, self.slider_y[5], "V max", self.v_max, 255)

        cv2.imshow(self.window_name, canvas)

    def get_mask(self, frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        lower = np.array([self.h_min, self.s_min, self.v_min], dtype=np.uint8)
        upper = np.array([self.h_max, self.s_max, self.v_max], dtype=np.uint8)

        return cv2.inRange(hsv, lower, upper)

    def process(self, frame):
        self.update_preview()

        mask = self.get_mask(frame)
        result = cv2.bitwise_and(frame, frame, mask=mask)

        return mask, result

    def destroy(self):
        cv2.destroyWindow(self.window_name)

In [22]:
import cv2
import numpy as np
import random
import time

detector = HSVColorDetector()

vid = cv2.VideoCapture(0, cv2.CAP_DSHOW)

vid.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
vid.set(cv2.CAP_PROP_FRAME_HEIGHT, 700)

GAME_WINDOW = "Juego de memorizacion"

cv2.namedWindow(GAME_WINDOW, cv2.WINDOW_NORMAL)
cv2.resizeWindow(GAME_WINDOW, 900, 700)

COLORS_DARK = [
    (40, 40, 120),      # Rojo oscuro
    (40, 110, 40),      # Verde oscuro
    (120, 70, 30),      # Azul oscuro
    (30, 120, 120)      # Amarillo/cian oscuro
]

COLORS_LIGHT = [
    (80, 80, 255),      # Rojo claro
    (80, 220, 80),      # Verde claro
    (255, 150, 70),     # Azul claro
    (80, 220, 220)      # Amarillo/cian claro
]

CONFIGURACION = 0
MOSTRAR_PATRON = 1
JUGADOR = 2
DERROTA = 3

state = CONFIGURACION

pattern = []
player_position = 0
pattern_position = 0

last_change = 0

LIGHT_TIME = 0.65
DARK_TIME = 0.25

MIN_OBJECT_AREA = 500

previous_position = None

MAX_TRACK_DISTANCE = 250

object_was_detected = False

last_selection_time = 0
SELECTION_DELAY = 0.25

FRAMES_NEEDED = 5

frames_inside = 0

MIN_OVERLAP = 0.45

rectangles = []


def update_rectangles(frame):
    global rectangles

    height, width = frame.shape[:2]

    margin_x = int(width * 0.05)
    gap = int(width * 0.02)

    total_width = width - 2 * margin_x

    rect_width = int((total_width - 3 * gap) / 4)
    rect_height = int(height * 0.20)

    start_x = margin_x
    start_y = int(height * 0.04)

    rectangles = []

    for i in range(4):
        x1 = start_x + i * (rect_width + gap)
        y1 = start_y

        x2 = x1 + rect_width
        y2 = y1 + rect_height

        rectangles.append((x1, y1, x2, y2))

def draw_rectangles(canvas, active=None):
    for i, (x1, y1, x2, y2) in enumerate(rectangles):
        if i == active:
            color = COLORS_LIGHT[i]
        else:
            color = COLORS_DARK[i]


        overlay = canvas.copy()

        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)

        canvas[:] = cv2.addWeighted(overlay, 0.65, canvas, 0.35, 0)

        cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 255, 255), 2)

        text = str(i + 1)
        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 2)[0]

        text_x = x1 + ((x2 - x1 - text_size[0]) // 2)
        text_y = y1 + ((y2 - y1 + text_size[1]) // 2)

        cv2.putText(canvas,
            text,(text_x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2, (255, 255, 255), 2)

def create_game_screen(frame, active=None, title="", subtitle=""):
    canvas = frame.copy()

    draw_rectangles(canvas, active)

    height, _ = canvas.shape[:2]

    if title:
        cv2.putText(canvas, title, (30, height - 80), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    if subtitle:
        cv2.putText(canvas,
            subtitle, (30, height - 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, (220, 220, 220), 1)

    return canvas

def get_object_detection(mask):
    global previous_position

    kernel = np.ones((5, 5), np.uint8)

    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates = []

    height, width = mask.shape[:2]

    max_area = width * height * 0.30

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < MIN_OBJECT_AREA:
            continue

        if area > max_area:
            continue

        perimeter = cv2.arcLength(contour,True)

        if perimeter <= 0:
            continue

        circularity = (4 * np.pi * area / (perimeter * perimeter))

        _, _, w, h = cv2.boundingRect(contour)

        aspect_ratio = w / float(h)

        if aspect_ratio < 0.25 or aspect_ratio > 4.0:
            continue

        hull = cv2.convexHull(contour)

        hull_area = cv2.contourArea(hull)

        if hull_area <= 0:
            continue

        solidity = area / hull_area

        (circle_x, circle_y), radius = (cv2.minEnclosingCircle(contour))

        cx = int(circle_x)
        cy = int(circle_y)

        shape_score = (circularity * 0.55 + solidity * 0.45)

        score = (area * (0.5 + shape_score))

        candidates.append({
            "score": score,
            "area": area,
            "cx": cx,
            "cy": cy,
            "radius": radius,
            "contour": contour,
            "circularity": circularity,
            "solidity": solidity
        })

    if not candidates:
        previous_position = None

        return None, mask_clean

    if previous_position is not None:
        previous_x, previous_y = previous_position

        nearby_candidates = []

        for candidate in candidates:
            dx = candidate["cx"] - previous_x
            dy = candidate["cy"] - previous_y

            distance = np.sqrt(dx * dx + dy * dy)

            if distance <= MAX_TRACK_DISTANCE:
                candidate["distance"] = distance

                nearby_candidates.append(candidate)

        if nearby_candidates:
            nearby_candidates.sort(key=lambda c: (c["distance"], -c["score"]))
            best = nearby_candidates[0]

        else:
            candidates.sort(key=lambda c: c["score"], reverse=True)
            best = candidates[0]

    else:
        candidates.sort(key=lambda c: c["score"], reverse=True)
        best = candidates[0]

    previous_position = (best["cx"], best["cy"])

    return best, mask_clean

def get_rectangle_from_contour(contour, mask_shape):
    if contour is None:
        return None, 0.0

    x, y, w, h = cv2.boundingRect(contour)

    if w <= 0 or h <= 0:
        return None, 0.0

    local_mask = np.zeros((h, w), dtype=np.uint8)

    shifted_contour = contour.copy()

    shifted_contour[:, :, 0] -= x
    shifted_contour[:, :, 1] -= y

    cv2.drawContours(local_mask, [shifted_contour], -1, 255, -1)

    object_pixels = cv2.countNonZero(local_mask)

    if object_pixels == 0:
        return None, 0.0

    best_rectangle = None
    best_overlap = 0.0

    for i, (x1, y1, x2, y2) in enumerate(rectangles):
        ix1 = max(x, x1)
        iy1 = max(y, y1)

        ix2 = min(x + w, x2)
        iy2 = min(y + h, y2)

        if ix1 >= ix2 or iy1 >= iy2:
            continue

        lx1 = ix1 - x
        ly1 = iy1 - y

        lx2 = ix2 - x
        ly2 = iy2 - y

        roi = local_mask[ly1:ly2, lx1:lx2]

        overlap_pixels = cv2.countNonZero(roi)

        overlap = (overlap_pixels / object_pixels)

        if overlap > best_overlap:
            best_overlap = overlap
            best_rectangle = i

    if best_overlap >= MIN_OVERLAP:
        return (best_rectangle, best_overlap)

    return None, best_overlap

def generate_next_pattern():
    pattern.append(random.randint(0, 3))

def start_pattern():
    global state
    global pattern_position
    global last_change
    global previous_position
    global frames_inside
    global object_was_detected

    pattern_position = 0
    last_change = time.time()

    previous_position = None
    frames_inside = 0
    object_was_detected = False

    state = MOSTRAR_PATRON

print("CONFIGURACIÓN DEL COLOR")
print("-----------------------")
print("Configura el color que utilizarás para el objeto.")
print("Cuando hayas terminado, pulsa ESPACIO para comenzar.")
print("Pulsa Q en cualquier momento para salir.")

try:
    while True:
        ret, frame = vid.read()

        if not ret:
            print("No se pudo obtener el fotograma.")
            break

        frame = cv2.flip(frame, 1)

        update_rectangles(frame)

        mask, result = detector.process(frame)

        cv2.imshow("Mascara", mask)
        cv2.imshow("Color detectado", result)

        detection, mask_clean = (get_object_detection(mask))

        detected_rectangle = None
        overlap = 0.0

        if detection is not None:
            object_x = detection["cx"]
            object_y = detection["cy"]

            object_area = detection["area"]

            contour = detection["contour"]

            detected_rectangle, overlap = (get_rectangle_from_contour(contour,mask_clean.shape))

        else:
            object_x = None
            object_y = None
            object_area = 0
            contour = None

        if state == CONFIGURACION:
            canvas = create_game_screen(
                frame,
                title="Configuracion del color",
                subtitle="Pulsa ESPACIO cuando hayas terminado"
            )

            cv2.putText(
                canvas,
                "Configura el color en la ventana HSV",(30, 330),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

            cv2.putText(canvas,
                "El objeto debe ser facilmente detectable",
                (30, 370), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (220, 220, 220), 1)

            cv2.imshow(GAME_WINDOW, canvas)

        elif state == MOSTRAR_PATRON:
            current_time = time.time()

            elapsed = (current_time - last_change)

            if elapsed < LIGHT_TIME:
                active = pattern[pattern_position]

                canvas = create_game_screen(
                    frame,
                    active=active,
                    title=f"Patron - Ronda {len(pattern)}",
                    subtitle="Memoriza la secuencia"
                )

            elif elapsed < (LIGHT_TIME + DARK_TIME):
                canvas = create_game_screen(
                    frame,
                    title=f"Patron - Ronda {len(pattern)}",
                    subtitle="Memoriza la secuencia"
                )

            else:
                pattern_position += 1

                last_change = current_time

                if pattern_position >= len(pattern):
                    state = JUGADOR

                    player_position = 0

                    object_was_detected = False

                    frames_inside = 0

                    previous_position = None

                    canvas = create_game_screen(frame, title="Tu turno",
                        subtitle="Repite el patron con el objeto")

                else:
                    canvas = create_game_screen(
                        frame, active=pattern[pattern_position],
                        title=f"Patron - Ronda {len(pattern)}",
                        subtitle="Memoriza la secuencia"
                    )

            cv2.imshow(GAME_WINDOW, canvas)

        elif state == JUGADOR:

            canvas = create_game_screen(frame,
                title=f"Tu turno - {player_position}/{len(pattern)}",
                subtitle="Mueve el objeto al rectangulo correspondiente"
            )

            if detection is not None:
                cv2.circle(canvas,
                    (object_x, object_y),
                    15,(255, 255, 255), 2)

                cv2.circle(canvas,
                    (object_x,object_y),
                    3, (255, 255, 255), -1)

                cv2.putText(canvas,
                    f"Area: {int(object_area)}",
                    (30, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,(255, 255, 255), 1)

            if detected_rectangle is not None:
                frames_inside += 1

                detected_x1, detected_y1, detected_x2, detected_y2 = (rectangles[detected_rectangle])

                cv2.rectangle(canvas,
                    (detected_x1, detected_y1),
                    (detected_x2, detected_y2),
                    (255, 255, 255), 4)

                cv2.putText(
                    canvas,
                    f"{int(overlap * 100)}%",
                    (detected_x1 + 10,
                    detected_y2 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2)

            else:
                frames_inside = 0
                object_was_detected = False

            if (detected_rectangle is not None and frames_inside >= FRAMES_NEEDED):

                current_time = time.time()

                if (not object_was_detected
                    and current_time - last_selection_time > SELECTION_DELAY):
                    selected = detected_rectangle

                    expected = pattern[player_position]

                    if selected == expected:
                        print(f"Correcto: rectangulo " f"{selected + 1}")

                        player_position += 1

                        last_selection_time = (current_time)

                        object_was_detected = True

                        frames_inside = 0

                        if (player_position >= len(pattern)):
                            generate_next_pattern()

                            print(f"Ronda " f"{len(pattern) - 1} " f"superada.")

                            pattern_position = 0

                            last_change = (time.time())

                            previous_position = None

                            state = MOSTRAR_PATRON

                    else:
                        print("Patron incorrecto.")

                        print(
                            f"Esperado: "
                            f"{expected + 1} | "
                            f"Seleccionado: "
                            f"{selected + 1}"
                        )

                        state = DERROTA

                        frames_inside = 0

            cv2.imshow(GAME_WINDOW, canvas)

        elif state == DERROTA:
            canvas = create_game_screen(frame, title="HAS PERDIDO", subtitle=f"Has llegado a la ronda {len(pattern)}")

            height, width = canvas.shape[:2]

            cv2.putText(canvas, "Pulsa R para volver a empezar", (30, height - 80), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(canvas, "Pulsa Q para salir", (30, height - 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

            cv2.imshow(GAME_WINDOW, canvas)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

        if (state == CONFIGURACION and key == ord(' ')):

            pattern = []

            generate_next_pattern()

            print("Comenzando partida...")
            print(f"Patron inicial: " f"{[x + 1 for x in pattern]}")

            start_pattern()

        elif (state == DERROTA and key == ord('r')):
            pattern = []
            generate_next_pattern()

            print("Nueva partida.")

            start_pattern()

finally:
    vid.release()
    cv2.destroyAllWindows()

CONFIGURACIÓN DEL COLOR
-----------------------
Configura el color que utilizarás para el objeto.
Cuando hayas terminado, pulsa ESPACIO para comenzar.
Pulsa Q en cualquier momento para salir.
Comenzando partida...
Patron inicial: [4]
Correcto: rectangulo 4
Ronda 1 superada.
Correcto: rectangulo 4
Correcto: rectangulo 4
Ronda 2 superada.
Correcto: rectangulo 4
Correcto: rectangulo 4
Correcto: rectangulo 4
Ronda 3 superada.
Correcto: rectangulo 4
Correcto: rectangulo 4
Correcto: rectangulo 4
Correcto: rectangulo 1
Ronda 4 superada.
Correcto: rectangulo 4
Correcto: rectangulo 4
Correcto: rectangulo 4
Patron incorrecto.
Esperado: 1 | Seleccionado: 4


Detección de bordes:

In [16]:
import cv2
import numpy as np
import time
import random

vid = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not vid.isOpened():
    print("ERROR: No se pudo abrir la cámara.")
    exit()

remove_background = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=False)

RECTANGLE_NUM = 4

pattern = [random.randint(1, RECTANGLE_NUM)]

pattern_position = 0

SHOWING_PATTERN = 0
WAITING = 1
CORRECT = 2
INCORRECT = 3
WON = 4

game_stage = SHOWING_PATTERN

start_time = time.time()

TIME_LIGHT = 0.7
TIME_BETWEEN = 0.25
TIME_SELECTION = 0.5
RESULT_TIME = 0.8

pattern_show_position = 0
pattern_show_stage = 0
pattern_timer = time.time()

time_detection = None
last_selection = None

waiting_for_release = False

position_history = []

POSITION_HISTORY_SIZE = 8

MIN_VALID_POSITIONS = 3

warmup_frames = 0
WARMUP_TIME = 60

def create_rectangles(width):
    margin = 30
    separation = 20

    available_x = (width - margin * 2 - separation * 3)

    rectangle_width = available_x // 4

    rectangles = []

    for i in range(4):
        x1 = (margin + i * (rectangle_width + separation))
        x2 = x1 + rectangle_width

        rectangles.append((x1, 40, x2, 160))

    return rectangles

def detect_player(mask):
    kernel = np.ones((7, 7), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return None

    valid_contours = []

    for contour in contours:
        area = cv2.contourArea(contour)

        if area >= 3000:
            valid_contours.append(contour)

    if len(valid_contours) == 0:
        return None

    contour = max(valid_contours, key=cv2.contourArea)

    area = cv2.contourArea(contour)

    x, y, w, h = cv2.boundingRect(contour)

    contour_points = contour[:, 0, :]

    top_y = np.min(contour_points[:, 1])

    top_threshold = top_y + max(15, int(h * 0.08))

    top_points = contour_points[contour_points[:, 1] <= top_threshold]

    if len(top_points) == 0:
        return None

    point_x = int(np.median(top_points[:, 0]))
    point_y = int(np.median(top_points[:, 1]))

    point = (point_x, point_y)

    return (point, (x, y, w, h), mask)

def stabilize_position(point):
    global position_history

    if point is None:
        return None

    position_history.append(point)

    if len(position_history) > POSITION_HISTORY_SIZE:
        position_history.pop(0)

    if len(position_history) < MIN_VALID_POSITIONS:
        return None

    x_values = [p[0] for p in position_history]
    y_values = [p[1] for p in position_history]

    stable_x = int(np.mean(x_values))
    stable_y = int(np.mean(y_values))

    return (stable_x, stable_y)

def detect_rectangle(point, rectangles):
    if point is None:
        return None

    px, py = point

    for i, rectangle in enumerate(rectangles):
        x1, y1, x2, y2 = rectangle

        if (x1 <= px <= x2 and y1 <= py <= y2):
            return i + 1

    return None

def get_pattern_rectangle():
    if pattern_show_position >= len(pattern):
        return None

    return pattern[pattern_show_position]


while True:
    ret, frame = vid.read()

    if not ret:
        print("ERROR: No se pudo leer la cámara.")
        break

    frame = cv2.flip(frame, 1)

    height, width, _ = frame.shape

    rectangles = create_rectangles(width)

    background_mask = remove_background.apply(frame)

    _, background_mask = cv2.threshold(background_mask, 200, 255, cv2.THRESH_BINARY)

    if warmup_frames < WARMUP_TIME:
        warmup_frames += 1

        player_point = None
        player_box = None

        position_history.clear()

    else:
        result = detect_player(background_mask)

        player_point = None
        player_box = None

        if result is not None:
            raw_point = result[0]
            player_box = result[1]
            background_mask = result[2]

            player_point = stabilize_position(raw_point)

    current_rectangle = detect_rectangle( player_point, rectangles)

    now = time.time()

    if game_stage == SHOWING_PATTERN:
        elapsed = now - pattern_timer

        if pattern_show_stage == 0:
            if elapsed >= TIME_LIGHT:
                pattern_show_stage = 1

                pattern_timer = now

        else:
            if elapsed >= TIME_BETWEEN:
                pattern_show_position += 1

                if (pattern_show_position >= len(pattern)):
                    game_stage = WAITING

                    pattern_position = 0
                    pattern_show_stage = 0

                    pattern_timer = now

                    last_selection = None
                    time_detection = None

                    waiting_for_release = False

                    position_history.clear()

                else:
                    pattern_show_stage = 0

                    pattern_timer = now

    elif game_stage == WAITING:
        if waiting_for_release:
            if current_rectangle is None:
                waiting_for_release = False

                last_selection = None
                time_detection = None

        else:
            if current_rectangle is not None:
                if current_rectangle != last_selection:
                    last_selection = current_rectangle

                    time_detection = now

                else:
                    if (time_detection is not None and now - time_detection >= TIME_SELECTION):
                        if (current_rectangle == pattern[pattern_position]):

                            pattern_position += 1

                            game_stage = CORRECT

                            start_time = now

                            waiting_for_release = True

                        else:
                            game_stage = INCORRECT

                            start_time = now
                            waiting_for_release = True

            else:
                last_selection = None
                time_detection = None

    elif game_stage == CORRECT:
        if now - start_time >= RESULT_TIME:
            if (pattern_position >= len(pattern)):
                pattern.append(random.randint(1, RECTANGLE_NUM))

                pattern_position = 0
                pattern_show_position = 0
                pattern_show_stage = 0

                pattern_timer = now

                game_stage = SHOWING_PATTERN

            else:
                game_stage = WAITING

                last_selection = None
                time_detection = None

    elif game_stage == INCORRECT:
        if now - start_time >= RESULT_TIME:
            pattern = [random.randint(1, RECTANGLE_NUM)]

            pattern_position = 0
            pattern_show_position = 0
            pattern_show_stage = 0

            pattern_timer = now

            game_stage = SHOWING_PATTERN

            last_selection = None
            time_detection = None

            waiting_for_release = False

    elif game_stage == WON:
        if now - start_time >= 3:
            pattern = [random.randint(1, RECTANGLE_NUM)]

            pattern_position = 0
            pattern_show_position = 0
            pattern_show_stage = 0
            pattern_timer = now

            game_stage = SHOWING_PATTERN

    for i, rect in enumerate(rectangles):
        number = i + 1

        x1, y1, x2, y2 = rect

        color = (255, 255, 255)

        thickness = 3

        if game_stage == SHOWING_PATTERN:
            active_rectangle = (get_pattern_rectangle())

            if number == active_rectangle:
                color = (0, 255, 255)

                thickness = 8

        elif (number == current_rectangle):
            if game_stage == WAITING:
                color = (0, 255, 255)

                thickness = 6

            elif game_stage == CORRECT:
                color = (0, 255, 0)

                thickness = 8

            elif game_stage == INCORRECT:
                color = (0, 0, 255)

                thickness = 8

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

        text = str(number)

        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 3)[0]

        tx = ( x1 + ( x2 - x1 - text_size[0]) // 2)
        ty = (y1 + ( y2 - y1 + text_size[1]) // 2)

        cv2.putText(frame, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)

    if player_box is not None:
        x, y, w, h = player_box

        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)

    if player_point is not None:
        cv2.circle(frame, player_point, 12, (0, 0, 255), -1)
        cv2.circle(frame, player_point, 20, (255, 255, 255), 2)

    if warmup_frames < WARMUP_TIME:
        cv2.putText(frame, "CALIBRANDO...", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    elif game_stage == SHOWING_PATTERN:
        cv2.putText(frame, "MEMORIZA", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        progress = (str(pattern_show_position + 1) + " / " + str(len(pattern)))

        cv2.putText(frame, progress, (30, 265), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    elif game_stage == WAITING:
        cv2.putText(frame, "REPITE EL PATRON", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        progress = (str(pattern_position) + " / " + str(len(pattern)))

        cv2.putText(frame, progress, (30, 265), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    elif game_stage == CORRECT:
        cv2.putText(frame, "CORRECTO!", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    elif game_stage == INCORRECT:
        cv2.putText(frame, "INCORRECTO!", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    elif game_stage == WON:
        cv2.putText(frame, "GANASTE!", (30, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(grey, 100, 200)

    player_edges = cv2.bitwise_and(edges, edges, mask=background_mask)

    cv2.imshow("Juego de Memorizacion", frame)

    cv2.imshow("Bordes del Jugador", player_edges)

    if cv2.waitKey(20) & 0xFF == 27:
        break

vid.release()

cv2.destroyAllWindows()